In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import random

MODEL_PATH = 'my_alzheimer_model.keras' 
DATA_DIR = './dataset' 
IMG_SIZE = 128

CLASS_NAMES = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
# ============================================

def load_and_preprocess_image_efficientnet(image_path):
    """Xử lý ảnh cho EfficientNet"""
    # 1. Đọc file
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    
    # 2. Resize
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    
    # 3. Preprocessing cho EfficientNet
    img_processed = tf.cast(img, tf.float32)
    
    
    return img_processed, img 

def predict_random_images(model, data_dir, num_images=9):
    plt.figure(figsize=(15, 15))
    
    # Lấy danh sách file ảnh
    all_image_paths = []
    all_image_labels = []
    
    # Duyệt qua các folder class
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir): continue
            
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_image_paths.append(os.path.join(class_dir, fname))
                all_image_labels.append(class_name)
    
    if len(all_image_paths) == 0:
        print("Không tìm thấy ảnh nào! Kiểm tra lại đường dẫn DATA_DIR.")
        return

    # Chọn ngẫu nhiên
    if len(all_image_paths) < num_images: num_images = len(all_image_paths)
    random_indices = random.sample(range(len(all_image_paths)), num_images)
    
    print(f"Đang dự đoán {num_images} ảnh ngẫu nhiên...")

    for i, idx in enumerate(random_indices):
        path = all_image_paths[idx]
        true_label = all_image_labels[idx]
        
        # Xử lý
        img_processed, img_original = load_and_preprocess_image_efficientnet(path)
        img_batch = tf.expand_dims(img_processed, axis=0) # Thêm chiều batch -> (1, 128, 128, 3)
        
        # Dự đoán
        predictions = model.predict(img_batch, verbose=0)
        
        # Lấy kết quả
        pred_idx = np.argmax(predictions[0])
        confidence = 100 * np.max(predictions[0])
        pred_label = CLASS_NAMES[pred_idx]
        
        # Vẽ hình
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(img_original.numpy().astype("uint8"))
        
        color = 'green' if true_label == pred_label else 'red'
        plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)", 
                  color=color, fontsize=11, fontweight='bold')
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# --- CHẠY ---
try:
    # Load model
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Load model EfficientNet thành công!")
    
    # Chạy
    predict_random_images(model, DATA_DIR)

except Exception as e:
    print(f"\nLỖI: {e}")
    print("Gợi ý: Kiểm tra lại đường dẫn file .keras và thư mục dataset.")